# NVDA 相对SOXX领先度与下一H5现金优势：必要秩检验

预先固定负号、来源、标签和2021–2022评价日历；只读封存Qlib DatasetH/Recorder，0拟合、0资金账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from scipy.stats import spearmanr

root=Path.cwd();card=root/"research/stage1_nvda_lead_cash_h5_rank_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["budget"]["new_market_GETs"]==plan["budget"]["new_model_fits"]==plan["budget"]["new_native_accounts"]==0
lead_card=root/src["lead_card"];dataset_card=root/src["dataset_card"]
assert sha(lead_card/"summary.json")==src["lead_summary_sha256"]
assert sha(dataset_card/"summary.json")==src["dataset_summary_sha256"]
ls=json.loads((lead_card/"summary.json").read_text());ds=json.loads((dataset_card/"summary.json").read_text())
assert ls["NAS_evidence"]["sha256"]==src["lead_NAS_sha256"] and ls["NAS_evidence"]["size_bytes"]==src["lead_NAS_size_bytes"]
assert ls["NAS_readback"]["verified"] and ls["NAS_readback"]["sha256"]==src["lead_NAS_sha256"]
assert ds["nas_evidence"]["sha256"]==src["dataset_NAS_sha256"] and ds["nas_evidence"]["size_bytes"]==src["dataset_NAS_size_bytes"]
assert ds["nas_readback"]["verified"] and ds["nas_readback"]["sha256"]==src["dataset_NAS_sha256"]
la=lead_card/"qlib_recorder"/src["lead_experiment_id"]/src["lead_recorder_id"]/"artifacts/analysis/lead"
da=dataset_card/"qlib_recorder"/src["dataset_experiment_id"]/src["dataset_recorder_id"]/"artifacts/inputs/dataset"
assert sha(la)==src["lead_artifact_sha256"] and sha(da)==src["dataset_artifact_sha256"]
assert sha(root/src["qlib_provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["qlib_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time="2021-01-01",end_time="2022-12-30",freq="day"))
assert len(calendar)==503
with R.uri_context(str(lead_card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["lead_experiment_id"],recorder_id=src["lead_recorder_id"])
 lead=rec.load_object("analysis/lead")
with R.uri_context(str(dataset_card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["dataset_experiment_id"],recorder_id=src["dataset_recorder_id"])
 dataset=rec.load_object("inputs/dataset")
train=dataset.prepare("train",col_set=["label","meta"])
assert len(train)==3020 and list(train["label"].columns)==["cash_advantage_H5","SOXL_relative_H5"]
dates=pd.DatetimeIndex(train.index.get_level_values("datetime"))
assert dates.is_unique and dates.is_monotonic_increasing
assert len(lead)==3966 and pd.DatetimeIndex(lead.index).is_unique
full=pd.DataFrame({"lead":lead.reindex(calendar).to_numpy(float)},index=calendar)
subset=train.loc[dates.isin(calendar)].copy()
subset.index=pd.DatetimeIndex(subset.index.get_level_values("datetime"))
assert subset.index.is_unique and subset.index.equals(calendar)
full["cash_advantage_H5"]=subset[("label","cash_advantage_H5")].astype(float)
full["train_mature"]=subset[("meta","train_mature")].astype(bool)
mask=full.train_mature & full.lead.notna() & full.cash_advantage_H5.notna()
assert int(mask.sum())>=plan["receiver"]["minimum_scored"]
assert full.lead.notna().all()
x=full.loc[mask,"lead"].to_numpy(float);y=full.loc[mask,"cash_advantage_H5"].to_numpy(float)
assert np.isfinite(x).all() and np.isfinite(y).all()
whole=float(spearmanr(x,y).statistic)
annual={}
for year in (2021,2022):
 pick=mask & (calendar.year==year)
 annual[str(year)]={"rows":int(pick.sum()),"Spearman":float(spearmanr(full.loc[pick,"lead"],full.loc[pick,"cash_advantage_H5"]).statistic)}
 assert annual[str(year)]["rows"]>=plan["receiver"]["annual_minimums"][str(year)]
fixed=plan["necessary_gate"]["stationary_block_upper_below_zero"]
rng=np.random.default_rng(fixed["seed"]);n=len(calendar);jump=1/fixed["mean_block_sessions"]
fx=full.lead.to_numpy(float);fy=full.cash_advantage_H5.where(mask).to_numpy(float)
rep=[]
for _ in range(fixed["replicates"]):
 idx=np.empty(n,dtype=int);idx[0]=rng.integers(n)
 for i in range(1,n):idx[i]=rng.integers(n) if rng.random()<jump else (idx[i-1]+1)%n
 xx,yy=fx[idx],fy[idx];valid=np.isfinite(xx)&np.isfinite(yy)
 if valid.sum()>=100:
  z=float(spearmanr(xx[valid],yy[valid]).statistic)
  if np.isfinite(z):rep.append(z)
assert len(rep)>=fixed["minimum_valid_replicates"]
upper=float(np.quantile(rep,1-fixed["one_sided_alpha"]))
gates={"full_negative":whole<0,"both_years_negative":all(a["Spearman"]<0 for a in annual.values()),"block_upper_below_zero":upper<0}
result={"status":"necessary_NVDA_lead_cash_H5_rank_pass" if all(gates.values()) else "not_promoted_fixed_NVDA_lead_cash_H5_rank_failed",
        "plan_sha256":sha(plan_path),"lead_recorder_id":src["lead_recorder_id"],"label_recorder_id":src["dataset_recorder_id"],
        "full_calendar_rows":n,"mature_scored_rows":len(x),"full_Spearman":whole,"annual":annual,
        "bootstrap":{"valid_replicates":len(rep),"mean_block_sessions":fixed["mean_block_sessions"],"one_sided_alpha":fixed["one_sided_alpha"],"upper_bound":upper},
        "gates":gates,"historical_first_public_qualified":False,"new_market_GETs":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":1,
        "SOXX_strategy_success":False,"independent_validation":False,
        "limits":["The H5 cash-over-SOXX price label is a necessary gross information target, not a funded Tiered account",
                  "NVDA relative price lead is not NVDA's SOXX constituent contribution and historical first-public price vintages remain unknown",
                  "The 2021-2022 time slice is reused development feedback, not independent strategy evidence"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":result["status"],"full":whole,"annual":annual,"block_upper":upper,"gates":gates})


{'status': 'not_promoted_fixed_NVDA_lead_cash_H5_rank_failed', 'full': -0.0074699068878877626, 'annual': {'2021': {'rows': 252, 'Spearman': 0.033837171547201}, '2022': {'rows': 245, 'Spearman': 0.012631682021069126}}, 'block_upper': 0.26178672738772657, 'gates': {'full_negative': True, 'both_years_negative': False, 'block_upper_below_zero': False}}


## 中文结论

原固定 NVDA-minus-SOXX 前20会话价格领先度来源曾在SOXL相对选择H5接收任务失败。本卡另冻一个**现金任务**必要条件：若领先反映未来SOXX持续走强，未来H5“现金相对SOXX优势”应随领先增强而降低，预定Spearman负号。只读封存Qlib来源与DatasetH标签，2021–2022完整503信号日日历保留，497行成熟可评分；2021为252、2022为245。0模型拟合、0资金账户、0新的市场请求。

全体相关 **−0.00747**，几乎为0；两个完整评价年分别为2021 **+0.03384**、2022 **+0.01263**，均与负号假说相反。固定63会话平均区块、10000次重采样及单侧校正分位数的相关上界 **+0.26179**，也未低于0。必要门仅全体负号点值勉强满足，逐年和稳定性均失败。

决定 `not_promoted_fixed_NVDA_lead_cash_H5_rank_failed`。不能利用2023狭窄AI龙头行情的事后叙事，反转该来源符号、挑年份或把失败的SOXL相对接收任务换成此现金版本继续拟合。NVDA相对价格并不等于其SOXX篮子贡献，历史Yahoo/Futu首次送达版本也未认证。该H5毛价格标签不是费后完整账户；SOXX v2目标未达成。
